In [ ]:
import cv2
import json
import matplotlib.pyplot as plt

id = "cocci_lactococcus_lactis_39_default_mmrsbpb_3_07102022115947"

filename = f"./train_data/images/{id}.jpg"

img = cv2.imread(filename)

with open("./inpaint_data_augmentation/circles.json", "r") as f:
    circle_data = json.load(f)
    circles = circle_data[id]

for circle in circles:
    cv2.circle(img, (circle["x"], circle["y"]), circle["r"], (100, 0, 255), 5)

cv2.imwrite(f"{id}_circles.jpg", img)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import re


def get_filename_components(filename: str) -> tuple[str, int, int, int]:
    """Extracts the components of a filename in the format id_x_y_r.ext"""

    match = re.search(r"(.+?)_(\d+)_(\d+)_(\d+)\.\w+", filename)

    if match:
        id = match.group(1)
        x, y, r = match.group(2), match.group(3), match.group(4)

        return id, int(x), int(y), int(r)
    else:
        raise ValueError(f"Filename {filename} does not match the expected pattern")

In [ ]:
from pathlib import Path
import numpy as np

patch_imgs = Path("./inpaint_data_augmentation/patched_images/correct").glob(
    f"{id}_*.jpg"
)

patch_imgs = [img for img in patch_imgs if img.stem.startswith(id)]


for img_path in patch_imgs:
    id, x, y, r = get_filename_components(img_path.name)

    cv2.circle(img, (x, y), r, (100, 255, 0), 5)


cv2.imwrite(f"{id}_circles_colored.jpg", img)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
from pathlib import Path
import numpy as np

patch_imgs = Path("./inpaint_data_augmentation/patched_images/correct").glob(
    f"{id}_*.jpg"
)

patch_imgs = [img for img in patch_imgs if img.stem.startswith(id)]

base_image = cv2.imread(filename)

for img in patch_imgs:
    id, x, y, r = get_filename_components(img.name)

    patch_img = cv2.imread(str(img))

    mask = np.zeros(patch_img.shape, dtype=np.uint8)
    cv2.circle(mask, (x, y), r, (255, 255, 255), -1)

    np.copyto(base_image, patch_img, where=(mask == (255, 255, 255)))


cv2.imwrite(f"{id}_patched.jpg", base_image)

plt.imshow(cv2.cvtColor(base_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()